In [2]:
# Initialize CustomRAG and process sample resumes
import sys
import os

# Set correct working directory
resume_parsing_dir = r"c:\Users\egyamah\ResumeParsing"
os.chdir(resume_parsing_dir)

# Add to Python path
if resume_parsing_dir not in sys.path:
    sys.path.insert(0, resume_parsing_dir)

from custom_rag import CustomRAG
from PyPDF2 import PdfReader

# Initialize CustomRAG with fresh database
db_path = "fresh_custom_rag_test.db"
rag = CustomRAG(db_path)

print("CustomRAG initialized with fresh database.")

# Process both sample resumes
data_path = os.path.join("data", "sample_resumes")
resume_files = ["ArjunResume-AIGEN.pdf", "DanielRESUMEAIgen.pdf"]

print("Processing sample resumes...")

for resume_file in resume_files:
    file_path = os.path.join(data_path, resume_file)
    
    if os.path.exists(file_path):
        print(f"Processing {resume_file}...")
        try:
            # Extract text directly using PyPDF2
            with open(file_path, 'rb') as f:
                reader = PdfReader(f)
                text = ""
                for page in reader.pages:
                    text += page.extract_text() + "\n"
            
            # Use text string with enter_resume_to_db
            metadata = {"filename": resume_file}
            result = rag.enter_resume_to_db(text, metadata)
            print(f"  Success: {result}")
            
        except Exception as e:
            print(f"  ERROR processing {resume_file}: {e}")
    else:
        print(f"  File not found: {file_path}")

print("Resume processing complete!")

# Check final count
count = rag.vector_db.client.get_or_create_collection('resumes').count()
print(f"Database now contains {count} resumes")

Using ChromaDB for vector similarity search
CustomRAG initialized with fresh database.
Processing sample resumes...
Processing ArjunResume-AIGEN.pdf...
✅ Extracted name: Arjun Mehta
✅ Extracted email: arjun.meh@gmail.com
✅ Extracted phone: (555) 123-4567
✅ Extracted name: Arjun Mehta
✅ Extracted email: arjun.meh@gmail.com
✅ Extracted phone: (555) 123-4567
  Success: Resume stored successfully in database with summary as main document
Processing DanielRESUMEAIgen.pdf...
  Success: Resume stored successfully in database with summary as main document
Processing DanielRESUMEAIgen.pdf...
✅ Extracted name: Daniel Cruz
✅ Extracted email: daniel.cruz.marketing@gmail.com
✅ Extracted phone: (555) 987-6543
✅ Extracted name: Daniel Cruz
✅ Extracted email: daniel.cruz.marketing@gmail.com
✅ Extracted phone: (555) 987-6543
  Success: Resume stored successfully in database with summary as main document
Resume processing complete!
Database now contains 4 resumes
  Success: Resume stored successfully in

In [3]:
# Test: Retrieve top resumes
job_desc = "Looking for Python developer with REST API experience."
top_resumes = rag.retrieve_top_resumes(job_desc, top_n=10)
print(top_resumes)

[{'rank': 1, 'candidate_id': '56f1480d-0fdd-4a9e-ad90-2fcf4c901066', 'summary': 'Arjun Mehta is a highly skilled software engineer with over 4 years of professional experience specializing in backend and full-stack development. With a strong proficiency in programming languages such as Java, Python, and Node.js, he has demonstrated expertise in designing, building, and optimizing scalable APIs and cloud-native applications. Arjun possesses in-depth knowledge of frameworks including Spring Boot and Flask, and has hands-on experience in developing microservices architectures, containerized systems, and distributed applications. His technical toolset extends to cloud platforms like AWS, containerization technologies such as Docker and Kubernetes, and databases including PostgreSQL and MongoDB. Adept at utilizing modern development workflows, Arjun is proficient in version control systems like Git, and continuous integration and deployment tools such as Jenkins and CI/CD pipelines. During 

In [4]:
# Test: Extract criteria from job description
criteria = rag.extract_criteria_from_jd(job_desc)
print(criteria)

### Resume Evaluation Criteria for Python Developer with REST API Experience at Ericsson  

#### Analysis:  
1. **Job Deliverables and Skills:**  
   - Core technical skills: Python development, REST API design and implementation.  
   - Secondary skills: Knowledge of scalable systems, cloud-native architectures, and integration best practices.  
   - Additional context: Focus on backend development, API performance optimization, and debugging.

2. **Business/Team Context:**  
   - Likely aligned with **BCSS** (Cloud-native platforms, network automation with AI/ML) or **BGCP** (Vonage APIs, programmable communications).  
   - Role likely emphasizes building reliable, scalable APIs for telecom-grade or SaaS use cases.  

3. **Role Function:**  
   - **Engineering**: Backend software development with a focus on APIs and cloud-based platforms.  

4. **Benchmarking:**  
   - **Industry leaders:** NVIDIA (high-performance backend systems), Google (API design), SAP (enterprise-grade softwar

In [11]:
# Working evaluation example with proper JSON output
import json

print("=== Complete Resume Evaluation Test ===")

job_desc = "Looking for Python developer with REST API experience."

# Check available resumes
all_resumes = rag.retrieve_top_resumes(job_desc, top_n=10)
available_count = len(all_resumes)
print(f"Available resumes in database: {available_count}")

# Use the actual available count
eval_results = rag.evaluate_top_resumes(job_desc, top_n=available_count)

print(f"\n✅ Successfully evaluated {len(eval_results)} resumes")
print("\n=== COMPLETE JSON OUTPUT ===")
print(json.dumps(eval_results, indent=2))

print("\n=== SUMMARY ===")
for i, result in enumerate(eval_results):
    if 'evaluation' in result and isinstance(result['evaluation'], dict):
        eval_data = result['evaluation']
        print(f"Resume {i+1}: {eval_data.get('recommendation', 'Unknown')} (Score: {eval_data.get('total_weighted_score', 'N/A')})")
    else:
        print(f"Resume {i+1}: {result.get('note', 'Unknown result format')}")

=== Complete Resume Evaluation Test ===
Available resumes in database: 4
Available resumes in database: 4
Evaluating 4 resumes with LLM...
DEBUG: Sending prompt preview:
You are an expert technical recruiter evaluating resumes for Ericsson positions.

**Task:** Evaluate multiple resumes against the provided job description.

**Data:**
{
  "job_description": "Looking for Python developer with REST API experience.",
  "resumes": [
    {
      "candidate_id": "56f1480d-0fdd-4a9e-ad90-2fcf4c901066",
      "resume": "\ud83e\udde0\n \n1.\n \nSoftware\n \nEngineer\n \n(SWE)\n \nName:\n \nArjun\n \nMehta\n \n \nEmail:\n \narjun.meh@gmail.com\n \n \nPhone:\n \n(555)\n \n...
Evaluating 4 resumes with LLM...
DEBUG: Sending prompt preview:
You are an expert technical recruiter evaluating resumes for Ericsson positions.

**Task:** Evaluate multiple resumes against the provided job description.

**Data:**
{
  "job_description": "Looking for Python developer with REST API experience.",
  "resumes": [